In [ ]:


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / 'Hispanic Chamber of Commerce'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


export=True


Chamber_H_1
- Table 1A

In [ ]:
path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
indicators = ['Chamber_H_1aa', 'Chamber_H_1ab']
geographies = ['Counties', 'MPO']
table_id = '1A'

list_df_geo = []

for geography in geographies:
    list_df_ind = []

    for indicator in indicators:
        workbook = f'{indicator} {geography} {estimate}.xlsx'
        sheet_name = geography

        if geography == 'Counties':
            geo_ID = ['County Name']
        if geography == 'MPO':
            geo_ID = ['MPO']


        ## Import ---

        file_in = path_in / workbook
        df_census = pd.read_excel(file_in, sheet_name=sheet_name)
        df_census = df_census[df_census['Year'] >= 2017]

        df_census = df_census[geo_ID + ['Year', 'Variable', 'Total']]
        df_census = df_census.pivot_table(index = geo_ID + ['Year']
                                            , columns = 'Variable'
                                            , values = 'Total').reset_index()
        df_census = df_census.sort_values(geo_ID + ['Year'], ascending = [item in geo_ID for item in geo_ID] + [False])
        df_census = df_census[geo_ID + ['Year', 'Under 18', 'Adult']]
        df_census = df_census.rename(columns = {'Under 18':f'Under 18_{indicator}', 'Adult':f'Adult_{indicator}'})
        list_df_ind.append(df_census)

    df_census = ft.reduce(lambda left, right: pd.merge(left, right, on = geo_ID + ['Year'], how = 'outer'), list_df_ind)
    df_census = df_census.sort_values(['Year'] + geo_ID, ascending=[False, True])
    df_census['Under 18_Chamber_H_1ab'] = df_census['Under 18_Chamber_H_1ab'] - df_census['Under 18_Chamber_H_1aa']
    df_census['Adult_Chamber_H_1ab'   ] = df_census['Adult_Chamber_H_1ab'   ] - df_census['Adult_Chamber_H_1aa'   ]
    indicator = indicator[:-2]
    df_census = df_census.rename(columns={'County Name':'Geography', 'MPO':'Geography'})
    list_df_geo.append(df_census)

df_census = pd.concat(list_df_geo)


df_census['Sort'] = pd.Categorical(df_census['Geography'], ['El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

df_census.columns = ['Geography', 'Year', 'Hispanic Under 18', 'Hispanic Adult', 'Non-Hispanic Under 18', 'Non-Hispanic Adult']
display(df_census)


## Exporting ---


if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)

Chamber_H_2
- Table 1B
- Table 1D


In [ ]:

path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
geographies =  'MPO'
indicator = 'Chamber_H_2'
geography = 'MPO'
table_id = '1B'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_census = pd.read_excel(file_in, sheet_name=sheet_name)

df_census = df_census[['MPO', 'Year', 'Variable', 'Total']]
df_census.loc[ df_census['Variable'].str.contains('Under 18'), 'Age Group'] = 'Children (under 18)'
df_census.loc[~df_census['Variable'].str.contains('Under 18'), 'Age Group'] = 'Adults (18+)'

df_census.loc[ df_census['Variable'].str.contains('born|Born'), 'Category'] = 'A'
df_census.loc[~df_census['Variable'].str.contains('born|Born'), 'Category'] = 'B'



def re_remove_post(x, exp = '('):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0].strip()
    
df_census['Variable'] = df_census['Variable'].apply(re_remove_post)


df_census['Percentage'] = df_census['Total'] / df_census.groupby(['MPO', 'Year', 'Category', 'Age Group'])['Total'].transform('sum')

                

df_census = df_census.sort_values(['Year', 'Category', 'Variable'], ascending=[False, False, False])


df_census1 = df_census.pivot_table(index = ['MPO', 'Year', 'Variable']
                                    , columns = 'Age Group'
                                    , values = 'Percentage').reset_index()
df_census1 = df_census1.rename(columns={'Adults (18+)':'Adults (18+)_pct', 'Children (under 18)':'Children (under 18)_pct'})
df_census2 = df_census.pivot_table(index = ['MPO', 'Year', 'Variable']
                                    , columns = 'Age Group'
                                    , values = 'Total').reset_index()
df_census = df_census1.merge(df_census2, on=['MPO', 'Year', 'Variable'])


df_census['Sort'] = pd.Categorical(df_census['Variable'], ['Born in the US'
                                                            , 'Not born in the US'
                                                            , 'US citizens'
                                                            , 'Not US citizens'
                                                        ])
df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)


display(df_census)



## Exporting ---
if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)




In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
geographies =  'Counties'
indicator = 'Chamber_H_2'
geography = 'Counties'
table_id = '1D'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)

df_counties = df_counties[['County Name', 'Year', 'Variable', 'Total']]
df_counties.loc[ df_counties['Variable'].str.contains('Under 18'), 'Age Group'] = 'Children (under 18)'
df_counties.loc[~df_counties['Variable'].str.contains('Under 18'), 'Age Group'] = 'Adults (18+)'

df_counties.loc[ df_counties['Variable'].str.contains('born|Born'), 'Category'] = 'A'
df_counties.loc[~df_counties['Variable'].str.contains('born|Born'), 'Category'] = 'B'



def re_remove_post(x, exp = '('):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0].strip()
    
df_counties['Variable'] = df_counties['Variable'].apply(re_remove_post)


df_counties['Percentage'] = df_counties['Total'] / df_counties.groupby(['County Name', 'Year', 'Category', 'Age Group'])['Total'].transform('sum')

                

df_counties = df_counties.sort_values(['Year', 'County Name', 'Category', 'Variable'], ascending=[False, True, False, False])


df_counties1 = df_counties.pivot_table(index = ['County Name', 'Year', 'Variable']
                                        , columns = 'Age Group'
                                        , values = 'Percentage').reset_index()
df_counties1 = df_counties1.rename(columns={'Adults (18+)':'Adults (18+)_pct', 'Children (under 18)':'Children (under 18)_pct'})
df_counties2 = df_counties.pivot_table(index = ['County Name', 'Year', 'Variable']
                                        , columns = 'Age Group'
                                        , values = 'Total').reset_index()
df_counties = df_counties1.merge(df_counties2, on=['County Name', 'Year', 'Variable'])


df_counties['Sort'] = pd.Categorical(df_counties['Variable'], ['Born in the US'
                                                                , 'Not born in the US'
                                                                , 'US citizens'
                                                                , 'Not US citizens'
                                                            ])
df_counties = df_counties.sort_values(['Year', 'County Name', 'Sort'], ascending=[False, True, True])
df_counties = df_counties.drop('Sort', axis=1)
df_counties = df_counties.reset_index(drop=True)






df_mpo = df_census.copy()
df_mpo      = df_mpo     .rename(columns={'MPO'        :'Geography'})
df_counties = df_counties.rename(columns={'County Name':'Geography'})

df_mpo = pd.concat([df_counties, df_mpo])

df_mpo['Population'] =  df_mpo['Adults (18+)'] + df_mpo['Children (under 18)']
df_mpo = df_mpo[['Geography', 'Variable', 'Year', 'Population']]

df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
                            , columns = 'Variable'
                            , values = 'Population').reset_index()

df_mpo = df_mpo[['Geography', 'Year', 'Born in the US', 'Not born in the US']]
df_mpo['Total'] = df_mpo['Born in the US'] + df_mpo['Not born in the US']
df_mpo['Born in the US_pct'    ] = df_mpo['Born in the US'    ]/df_mpo['Total']
df_mpo['Not born in the US_pct'] = df_mpo['Not born in the US']/df_mpo['Total']

df_mpo = df_mpo[['Geography', 'Year', 'Born in the US', 'Born in the US_pct', 'Not born in the US', 'Not born in the US_pct']]


df_mpo['Sort'] = pd.Categorical(df_mpo['Geography'], ['El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_mpo = df_mpo.sort_values(['Year', 'Sort'], ascending=[False, True])
df_mpo = df_mpo.drop('Sort', axis=1)
df_mpo = df_mpo.reset_index(drop=True)

display(df_mpo)





## Exporting ---
if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_5
- Table 2A

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
indicator = 'Chamber_H_5'
geography = 'Counties'
table_id = '2A'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
df_counties = df_counties.rename(columns={'County Name':'Geography'})
df_counties = df_counties.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'Race_Ethnicity'
                                        , values = 'Total').reset_index()


df_counties

geography = 'MPO'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'Race_Ethnicity'
                                        , values = 'Total').reset_index()


df_census = pd.concat([df_counties, df_mpo])

df_census['Sort'] = pd.Categorical(df_census['Geography'], [
                                                        'El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                   ])

df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

display(df_census)




## Exporting ---
if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)



In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
indicator = 'Chamber_H_6'
geography = 'Counties'
table_id = '2B'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = 'PUMA'

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
df_counties = df_counties.rename(columns={'County Name':'Geography'})
df_counties['GRPIP'] = df_counties['GRPIP'].astype(int)
df_counties = df_counties[df_counties['GRPIP'] != 0]
wm = lambda x: np.average(x, weights = df_counties.loc[x.index, "Households"])/100 # weighted average
df_counties = df_counties.groupby(['Geography', 'Year', 'HISP'], as_index=False).agg(GRPIP = ('GRPIP', wm))

df_counties = df_counties.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'HISP'
                                        , values = 'GRPIP').reset_index()



geography = 'MPO'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo['GRPIP'] = df_mpo['GRPIP'].astype(int)
df_mpo = df_mpo[df_mpo['GRPIP'] != 0]
wm = lambda x: np.average(x, weights = df_mpo.loc[x.index, "Households"])/100 # weighted average
df_mpo = df_mpo.groupby(['Geography', 'Year', 'HISP'], as_index=False).agg(GRPIP = ('GRPIP', wm))

df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'HISP'
                                        , values = 'GRPIP').reset_index()


df_census = pd.concat([df_counties, df_mpo])

df_census['Sort'] = pd.Categorical(df_census['Geography'], [
                                                        'El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

display(df_census)




## Exporting ---
if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)


Chamber_H_7

Chamber_H_8

In [ ]:


table_id = '3A'


url_edu24 = "https://www3.cde.ca.gov/demo-downloads/acgr/acgr24.txt"
df_edu24 = pd.read_csv(url_edu24, sep = '\t')
# display(df_edu24)


url_edu22 = "https://www3.cde.ca.gov/demo-downloads/acgr/acgr19.txt"
df_edu22 = pd.read_csv(url_edu22, sep = '\t')
# display(df_edu22)

df_edu = pd.concat([df_edu24, df_edu22])
print(df_edu.columns)

df_edu = df_edu[df_edu['ReportingCategory'].isin(['RH', 'RW'])]
df_edu = df_edu[df_edu['AggregateLevel'] == 'C']
df_edu = df_edu[df_edu['CountyName'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
df_edu['CohortStudents'                      ] = df_edu['CohortStudents'                      ].replace('*', '0').astype(int)
df_edu['Regular HS Diploma Graduates (Count)'] = df_edu['Regular HS Diploma Graduates (Count)'].replace('*', '0').astype(int)
df_edu["Met UC/CSU Grad Req's (Count)"] = df_edu["Met UC/CSU Grad Req's (Count)"].replace('*', '0').astype(int)

df_edu1 = df_edu.groupby(['AcademicYear', 'CountyName', 'ReportingCategory'], as_index=False).agg(TotalStudents = ('CohortStudents', 'sum'), Graduated = ('Regular HS Diploma Graduates (Count)', 'sum'), Met_UC_req = ("Met UC/CSU Grad Req's (Count)", 'sum'))
df_edu2 = df_edu.groupby(['AcademicYear'              , 'ReportingCategory'], as_index=False).agg(TotalStudents = ('CohortStudents', 'sum'), Graduated = ('Regular HS Diploma Graduates (Count)', 'sum'), Met_UC_req = ("Met UC/CSU Grad Req's (Count)", 'sum'))
df_edu2['CountyName']='SACOG'

df_edu = pd.concat([df_edu1, df_edu2])

df_edu['Graduated_pct' ] = df_edu['Graduated' ]/df_edu['TotalStudents']
df_edu['Met_UC_req_pct'] = df_edu['Met_UC_req']/df_edu['Graduated'    ]

df_edu1 = df_edu.pivot_table(index=['AcademicYear', 'CountyName'], columns='ReportingCategory', values='Graduated_pct').reset_index()
df_edu1 = df_edu1.rename(columns={'RH':'Graduating Hispanic_pct', 'RW':'Graduating White_pct'})

df_edu2 = df_edu.pivot_table(index=['AcademicYear', 'CountyName'], columns='ReportingCategory', values='Met_UC_req_pct').reset_index()
df_edu2 = df_edu2.rename(columns={'RH':'Met_UC_req Hispanic_pct', 'RW':'Met_UC_req White_pct'})

df_edu = df_edu1.merge(df_edu2)


df_edu['Sort'] = pd.Categorical(df_edu['CountyName'], ['El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_edu = df_edu.sort_values(['AcademicYear', 'Sort'], ascending=[False, True])
df_edu = df_edu.drop('Sort', axis=1)
df_edu = df_edu.reset_index(drop=True)

display(df_edu)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_edu.to_excel(writer, sheet_name=table_id, index=False)




Chamber_H_9

In [ ]:

path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies =  'MPO'
indicator = 'Chamber_H_9'
geography = 'MPO'
table_id = '3B'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_edu = pd.read_excel(file_in, sheet_name=sheet_name)
df_edu = df_edu.drop('Percentage', axis=1)
df_edu['AGEP'      ] = df_edu['AGEP'      ].astype(int)
df_edu['Population'] = df_edu['Population'].astype(int)
df_edu

df_edu = df_edu[df_edu['AGEP'] >= 25]

conditions = [
      (df_edu['AGEP'] >= 25) & (df_edu['AGEP'] <= 40)
    , (df_edu['AGEP'] >= 41) & (df_edu['AGEP'] <= 64)
    , (df_edu['AGEP'] >= 65)
    ]

choices = ['Age 25-40', 'Age 41-64', 'Age 65+']

df_edu['Age group'] = np.select(conditions, choices, default='no')

df_edu  = df_edu.groupby(['Year', 'HISP', 'Age group', 'SCHL'], as_index=False).agg(Population = ('Population', 'sum'))
df_edu2 = df_edu.groupby(['Year', 'HISP',              'SCHL'], as_index=False).agg(Population = ('Population', 'sum'))

df_edu2['Age group'] = 'Total'

df_edu = pd.concat([df_edu, df_edu2])

df_edu['Percentage'] = df_edu['Population']/df_edu.groupby(['Year', 'HISP', 'Age group'])['Population'].transform('sum')



df_edu = df_edu.pivot_table(index=['Year', 'Age group', 'SCHL'], columns='HISP', values='Percentage').reset_index()

df_edu['Sort'] = pd.Categorical(df_edu['SCHL'], [
    'Less than HS'
    , 'HS or equivalent'
    , 'Some college/AA'
    , 'BA/BS'
    , 'Graduate'
])

df_edu = df_edu.sort_values(['Year', 'Age group', 'Sort'], ascending=[False, True, True])
df_edu = df_edu.drop('Sort', axis=1)
df_edu = df_edu.reset_index(drop=True)

display(df_edu)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_edu.to_excel(writer, sheet_name=table_id, index=False)

